# EDA
create: 2026-03-25

In [1]:
import os, sys
from dotenv import load_dotenv
import glob
import zipfile
import requests
import pandas as pd
from io import BytesIO
from datetime import datetime, timedelta, timezone
from typing import List, Dict
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from neo4j import GraphDatabase

In [4]:
load_dotenv("../.env")

DATA_DIR = "../kgdata"
ENDPOINT_URL = "http://data.gdeltproject.org/gdeltv2"
MASTER_LIST_URL = f"{ENDPOINT_URL}/masterfilelist.txt"

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

DAYS = 1

In [5]:
EVENT_COLS = [
    "GlobalEventID","Day","MonthYear","Year","FractionDate",
    "Actor1Code","Actor1Name","Actor1CountryCode","Actor1KnownGroupCode",
    "Actor1EthnicCode","Actor1Religion1Code","Actor1Religion2Code",
    "Actor1Type1Code","Actor1Type2Code","Actor1Type3Code",
    "Actor2Code","Actor2Name","Actor2CountryCode","Actor2KnownGroupCode",
    "Actor2EthnicCode","Actor2Religion1Code","Actor2Religion2Code",
    "Actor2Type1Code","Actor2Type2Code","Actor2Type3Code",
    "IsRootEvent","EventCode","EventBaseCode","EventRootCode",
    "QuadClass","GoldsteinScale","NumMentions","NumSources",
    "NumArticles","AvgTone",
    "Actor1Geo_Type","Actor1Geo_FullName","Actor1Geo_CountryCode",
    "Actor1Geo_ADM1Code","Actor1Geo_ADM2Code",
    "Actor1Geo_Lat","Actor1Geo_Long","Actor1Geo_FeatureID",
    "Actor2Geo_Type","Actor2Geo_FullName","Actor2Geo_CountryCode",
    "Actor2Geo_ADM1Code","Actor2Geo_ADM2Code",
    "Actor2Geo_Lat","Actor2Geo_Long","Actor2Geo_FeatureID",
    "ActionGeo_Type","ActionGeo_FullName","ActionGeo_CountryCode",
    "ActionGeo_ADM1Code","ActionGeo_ADM2Code",
    "ActionGeo_Lat","ActionGeo_Long","ActionGeo_FeatureID",
    "DATEADDED","SOURCEURL",
]
 
MENTION_COLS = [
    "GlobalEventID","EventTimeDate","MentionTimeDate",
    "MentionType","MentionSourceName","MentionIdentifier",
    "SentenceID","Actor1CharOffset","Actor2CharOffset",
    "ActionCharOffset","InRawText","Confidence",
    "MentionDocLen","MentionDocTone","MentionDocTranslationInfo","Extras",
]
 
GKG_COLS = [
    "GKGRECORDID","DATE","SourceCollectionIdentifier","SourceCommonName",
    "DocumentIdentifier","Counts","V2Counts","Themes","V2Themes",
    "Locations","V2Locations","Persons","V2Persons",
    "Organizations","V2Organizations","V2Tone","Dates",
    "GCAM","SharingImage","RelatedImages","SocialImageEmbeds",
    "SocialVideoEmbeds","Quotations","AllNames","Amounts",
    "TranslationInfo","Extras",
]
 
CAMEO_MAP = {
    "01":"MAKE_PUBLIC_STATEMENT","02":"APPEAL",
    "03":"EXPRESS_INTENT_TO_COOPERATE","04":"CONSULT",
    "05":"ENGAGE_IN_DIPLOMACY","06":"PROVIDE_MATERIAL_COOPERATION",
    "07":"PROVIDE_AID","08":"YIELD","09":"INVESTIGATE",
    "10":"DEMAND","11":"DISAPPROVE","12":"REJECT",
    "13":"THREATEN","14":"PROTEST","15":"EXHIBIT_FORCE",
    "16":"REDUCE_RELATIONS","17":"COERCE","18":"ASSAULT",
    "19":"FIGHT","20":"USE_UNCONVENTIONAL_MASS_VIOLENCE",
}
 
QUAD_MAP = {
    "1":"VERBAL_COOPERATION","2":"MATERIAL_COOPERATION",
    "3":"VERBAL_CONFLICT","4":"MATERIAL_CONFLICT",
}

## Download GDelt Data

In [6]:
EXTENSIONS = ["export.CSV", "mentions.CSV", "gkg.csv"]

def get_timestamps(days: int) -> List[str]:
    rows = requests.get(MASTER_LIST_URL, timeout=30).text.strip().split('\n')
    cutoff = datetime.now(timezone.utc) - timedelta(days=days)

    timestamps = []
    for row in rows:
        if not row.endswith(".gkg.csv.zip"):
            continue
        url = row.split()[-1]
        ts_str = url.split('/')[-1].split('.')[0]
        ts = datetime.strptime(ts_str, "%Y%m%d%H%M%S").replace(tzinfo=timezone.utc)

        if ts >= cutoff:
            timestamps.append(ts_str)

    return timestamps

def download_and_extract(ts: str) -> List[str]:
    extracted = []
    for ext in EXTENSIONS:
        output_path = os.path.join(DATA_DIR, f"{ts}.{ext}")
        if os.path.exists(output_path):
            extracted.append(output_path)
            continue
        try:
            resp = requests.get("f{ENDPOINT_URL}/{ts}.{ext}.zip", timeout=30)
            resp.raise_for_status()
            with zipfile.ZipFile(BytesIO(resp.content)) as zf:
                for member in zf.namelist():
                    target = os.path.join(DATA_DIR, member)
                    if not os.path.exists(target):
                        zf.extract(member, DATA_DIR)
                    extracted.append(target)
        except Exception:
            pass

    return extracted

In [7]:
timestamps = get_timestamps(DAYS)
all_files = []
with ThreadPoolExecutor(max_workers=8) as pool:
    futures = {pool.submit(download_and_extract, ts): ts for ts in timestamps}
    for f in tqdm(as_completed(futures), total=len(futures), desc="download"):
        all_files.extend(f.result())

download:   0% 0/97 [00:00<?, ?it/s]

download: 100% 97/97 [00:00<00:00, 5276.61it/s]

## CSV Load

In [8]:
def load_table(pattern: str, columns: List[str]) -> pd.DataFrame:
    files = sorted(glob.glob(os.path.join(DATA_DIR, pattern)))
    if not files:
        return pd.DataFrame(columns=columns)

    chunks = []
    for f in files:
        try:
            df = pd.read_csv(
                f, sep='\t', header=None, dtype=str,
                on_bad_lines="skip", engine="python",
            )

            if len(df.columns) >= len(columns):
                df = df.iloc[:, :len(columns)]
            df.columns = columns[:len(df.columns)]
            chunks.append(df)
        except Exception as e:
            print(f"{os.path.basename(f)}: {e}")

    if not chunks:
        return pd.DataFrame(columns=columns)

    return pd.concat(chunks, ignore_index=True)

In [9]:
events_raw = load_table("*.export.CSV", EVENT_COLS)
mentions_raw = load_table("*.mentions.CSV", MENTION_COLS)
gkg_raw = load_table("*.gkg.csv", GKG_COLS)

print(f"Events: {len(events_raw):>10,}")
print(f"Mentions: {len(mentions_raw):>10,}")
print(f"GKG: {len(gkg_raw):>10,}")

20260324131500.gkg.csv: 'utf-8' codec can't decode byte 0xa9 in position 1169: invalid start byte


Events:    123,549
Mentions:    365,649
GKG:    129,046


## Event Preprocessing


In [10]:
events = events_raw.copy()
events = events.dropna(subset=["Actor1Code", "Actor2Code"], how="all")
events["Actor1Code"] = events["Actor1Code"].fillna("UNKNOWN")
events["Actor2Code"] = events["Actor2Code"].fillna("UNKNOWN")
events["Actor1Name"] = events["Actor1Name"].fillna("UNKNOWN")
events["Actor2Name"] = events["Actor2Name"].fillna("UNKNOWN")

for col in ["GoldsteinScale", "AvgTone", "NumMentions", "NumSources", "NumArticles"]:
    events[col] = pd.to_numeric(events[col], errors="coerce").fillna(0)

events["EventRootCode"] = events["EventRootCode"].fillna(events["EventCode"].str[:2])
events["RelationType"]  = events["EventRootCode"].map(CAMEO_MAP).fillna("UNKNOWN_EVENT")
events["QuadClassName"]  = events["QuadClass"].map(QUAD_MAP).fillna("UNKNOWN")
 
events["EventDate"] = pd.to_datetime(events["Day"], format="%Y%m%d", errors="coerce")
events["DateStr"]   = events["EventDate"].dt.strftime("%Y-%m-%d").fillna("1970-01-01")
 
events = events.drop_duplicates(subset=["GlobalEventID"])

## Mentions Preprocessing

In [11]:
mentions = mentions_raw.copy()
mentions["Confidence"]     = pd.to_numeric(mentions["Confidence"], errors="coerce").fillna(0)
mentions["MentionDocTone"] = pd.to_numeric(mentions["MentionDocTone"], errors="coerce").fillna(0)
 
valid_ids = set(events["GlobalEventID"].astype(str))
mentions = mentions[mentions["GlobalEventID"].astype(str).isin(valid_ids)]
mentions = mentions[mentions["MentionSourceName"].notna()]

## GKG Preprocessing

In [12]:
gkg = gkg_raw.drop_duplicates(subset=["GKGRECORDID"]).copy()
 
gkg["_themes"]  = gkg["V2Themes"].fillna(gkg["Themes"])
gkg["_persons"] = gkg["V2Persons"].fillna(gkg["Persons"])
gkg["_orgs"]    = gkg["V2Organizations"].fillna(gkg["Organizations"])
gkg["_locs"]    = gkg["V2Locations"].fillna(gkg["Locations"])
gkg["Tone"] = gkg["V2Tone"].str.split(",").str[0].pipe(pd.to_numeric, errors="coerce").fillna(0)

In [13]:
def explode_gkg_field(
    df: pd.DataFrame, src_col: str, new_col: str,
    keep_cols: List[str] = ["SourceCommonName", "DocumentIdentifier", "Tone"]) -> pd.DataFrame:
    sub = df[df[src_col].notna()][keep_cols + [src_col]].copy()
    if sub.empty:
        return pd.DataFrame(columns=keep_cols + [new_col])
    
    sub[new_col] = sub[src_col].str.split(";").apply(
        lambda parts: [p.split(",")[0].strip() for p in parts if p.strip()]
    )
    sub = sub.drop(columns=[src_col]).explode(new_col)
    sub = sub[sub[new_col].astype(bool)]
    return sub.reset_index(drop=True)

themes_df  = explode_gkg_field(gkg, "_themes", "Theme")
persons_df = explode_gkg_field(gkg, "_persons", "Person")
orgs_df    = explode_gkg_field(gkg, "_orgs", "Org")

In [14]:
def parse_locations(df: pd.DataFrame) -> pd.DataFrame:
    sub = df[df["_locs"].notna()][["SourceCommonName", "_locs"]].copy()
    if sub.empty:
        return pd.DataFrame()
 
    records = []
    for _, row in sub.iterrows():
        source = row["SourceCommonName"]
        for entry in str(row["_locs"]).split(";"):
            parts = entry.split("#")
            if len(parts) >= 7 and parts[1].strip():
                try:
                    records.append({
                        "SourceCommonName": source,
                        "LocName": parts[1].strip(),
                        "CountryCode": parts[2].strip(),
                        "Lat": float(parts[5]) if parts[5].strip() else None,
                        "Long": float(parts[6]) if parts[6].strip() else None,
                    })
                except (ValueError, IndexError):
                    continue
 
    result = pd.DataFrame(records)
    return result
 
 
locs_df = parse_locations(gkg)

## Neo4j
MATCH (n) DETACH DELETE n;

In [15]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
 
with driver.session(database=NEO4J_DATABASE) as s:
    result = s.run("RETURN 1 AS ok").single()
 
SCHEMA_QUERIES = [
    "CREATE CONSTRAINT IF NOT EXISTS FOR (a:Actor) REQUIRE a.code IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (e:Event) REQUIRE e.global_event_id IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (t:Theme) REQUIRE t.name IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (p:Person) REQUIRE p.name IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (o:Organization) REQUIRE o.name IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (l:Location) REQUIRE l.name IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (s:Source) REQUIRE s.name IS UNIQUE",
    "CREATE INDEX IF NOT EXISTS FOR (a:Actor) ON (a.country_code)",
    "CREATE INDEX IF NOT EXISTS FOR (e:Event) ON (e.date)",
]
 
with driver.session() as s:
    for q in SCHEMA_QUERIES:
        s.run(q)

In [19]:
def batch_load(cypher: str, records: list, batch_size: int = 2000, label: str = ""):
    total = len(records)
    if total == 0:
        print(f"  ⚠ {label}: 데이터 없음")
        return
 
    with driver.session() as session:
        for i in tqdm(range(0, total, batch_size), desc=label):
            batch = records[i : i + batch_size]
            session.run(cypher, batch=batch)

    print(f" {label}: {total:,} Load Completed")

In [ ]:
### Nodes

In [20]:
CYPHER_EVENTS = """
UNWIND $batch AS r
MERGE (a1:Actor {code: r.a1_code})
  ON CREATE SET a1.name = r.a1_name, a1.country_code = r.a1_country, a1.type1 = r.a1_type
  ON MATCH SET  a1.name = CASE WHEN r.a1_name <> 'UNKNOWN' THEN r.a1_name ELSE a1.name END
MERGE (a2:Actor {code: r.a2_code})
  ON CREATE SET a2.name = r.a2_name, a2.country_code = r.a2_country, a2.type1 = r.a2_type
  ON MATCH SET  a2.name = CASE WHEN r.a2_name <> 'UNKNOWN' THEN r.a2_name ELSE a2.name END
CREATE (a1)-[:EVENT {
    global_event_id: r.eid,
    cameo_code:      r.cameo,
    relation:        r.relation,
    quad_class:      r.quad,
    goldstein:       r.goldstein,
    tone:            r.tone,
    date:            date(r.date_str),
    num_mentions:    r.mentions,
    source_url:      r.url
}]->(a2)
"""
 
event_records = events.rename(columns={
    "Actor1Code":"a1_code", "Actor1Name":"a1_name",
    "Actor1CountryCode":"a1_country", "Actor1Type1Code":"a1_type",
    "Actor2Code":"a2_code", "Actor2Name":"a2_name",
    "Actor2CountryCode":"a2_country", "Actor2Type1Code":"a2_type",
    "GlobalEventID":"eid", "EventCode":"cameo",
    "RelationType":"relation", "QuadClassName":"quad",
    "GoldsteinScale":"goldstein", "AvgTone":"tone",
    "DateStr":"date_str", "NumMentions":"mentions", "SOURCEURL":"url",
})[["a1_code","a1_name","a1_country","a1_type",
    "a2_code","a2_name","a2_country","a2_type",
    "eid","cameo","relation","quad","goldstein","tone","date_str","mentions","url"
]].fillna("").to_dict("records")
 
batch_load(CYPHER_EVENTS, event_records, label="Events")

Events:   0% 0/62 [00:00<?, ?it/s]

Events:   2% 1/62 [00:00<00:12,  4.77it/s]

Events:   5% 3/62 [00:00<00:06,  8.62it/s]

Events:   8% 5/62 [00:00<00:05, 10.12it/s]

Events:  11% 7/62 [00:00<00:05, 10.77it/s]

...

Events:  92% 57/62 [00:05<00:00, 10.97it/s]

Events:  95% 59/62 [00:05<00:00, 10.91it/s]

Events:  98% 61/62 [00:05<00:00, 11.00it/s]

Events: 100% 62/62 [00:05<00:00, 11.09it/s]

 Events: 123,549 Load Completed


In [21]:
CYPHER_THEMES = """
UNWIND $batch AS r
MERGE (s:Source {name: r.source})
  ON CREATE SET s.url = r.url
MERGE (t:Theme {name: r.theme})
MERGE (s)-[rel:HAS_THEME]->(t)
  ON CREATE SET rel.tone = r.tone, rel.count = 1
  ON MATCH SET  rel.count = rel.count + 1
"""
 
if not themes_df.empty:
    t_records = themes_df.rename(columns={
        "SourceCommonName":"source", "DocumentIdentifier":"url",
        "Theme":"theme", "Tone":"tone",
    })[["source","url","theme","tone"]].fillna("").to_dict("records")
    batch_load(CYPHER_THEMES, t_records, label="Themes")

Themes:   0% 0/2746 [00:00<?, ?it/s]

Themes:   0% 1/2746 [00:00<12:59,  3.52it/s]

Themes:   0% 3/2746 [00:00<05:57,  7.68it/s]

Themes:   0% 5/2746 [00:00<04:10, 10.94it/s]

Themes:   0% 7/2746 [00:00<03:28, 13.12it/s]

...

Themes: 100% 2741/2746 [02:16<00:00, 15.66it/s]

Themes: 100% 2743/2746 [02:16<00:00, 15.55it/s]

Themes: 100% 2745/2746 [02:17<00:00, 15.81it/s]

Themes: 100% 2746/2746 [02:17<00:00, 20.03it/s]

 Themes: 5,491,977 Load Completed


In [22]:
CYPHER_THEMES = """
UNWIND $batch AS r
MERGE (s:Source {name: r.source})
  ON CREATE SET s.url = r.url
MERGE (t:Theme {name: r.theme})
MERGE (s)-[rel:HAS_THEME]->(t)
  ON CREATE SET rel.tone = r.tone, rel.count = 1
  ON MATCH SET  rel.count = rel.count + 1
"""
 
if not themes_df.empty:
    t_records = themes_df.rename(columns={
        "SourceCommonName":"source", "DocumentIdentifier":"url",
        "Theme":"theme", "Tone":"tone",
    })[["source","url","theme","tone"]].fillna("").to_dict("records")
    batch_load(CYPHER_THEMES, t_records, label="Themes")

Themes:   0% 0/2746 [00:00<?, ?it/s]

Themes:   0% 1/2746 [00:00<05:20,  8.57it/s]

Themes:   0% 3/2746 [00:00<03:29, 13.12it/s]

Themes:   0% 5/2746 [00:00<03:13, 14.19it/s]

Themes:   0% 7/2746 [00:00<03:03, 14.89it/s]

...

Themes: 100% 2742/2746 [02:39<00:00, 15.38it/s]

Themes: 100% 2744/2746 [02:40<00:00, 15.89it/s]

Themes: 100% 2746/2746 [02:40<00:00, 15.92it/s]

Themes: 100% 2746/2746 [02:40<00:00, 17.15it/s]

 Themes: 5,491,977 Load Completed


In [23]:
CYPHER_ORGS = """
UNWIND $batch AS r
MERGE (s:Source {name: r.source})
MERGE (o:Organization {name: r.org})
MERGE (s)-[rel:MENTIONS_ORG]->(o)
  ON CREATE SET rel.count = 1
  ON MATCH SET  rel.count = rel.count + 1
"""
 
if not orgs_df.empty:
    o_records = orgs_df.rename(columns={
        "SourceCommonName":"source", "Org":"org",
    })[["source","org"]].fillna("").to_dict("records")
    batch_load(CYPHER_ORGS, o_records, label="Organizations")

Organizations:   0% 0/229 [00:00<?, ?it/s]

Organizations:   1% 2/229 [00:00<00:13, 16.69it/s]

Organizations:   3% 6/229 [00:00<00:08, 26.35it/s]

Organizations:   4% 10/229 [00:00<00:07, 31.06it/s]

Organizations:   6% 14/229 [00:00<00:06, 32.61it/s]

...

Organizations:  95% 218/229 [00:06<00:00, 30.91it/s]

Organizations:  97% 222/229 [00:06<00:00, 31.12it/s]

Organizations:  99% 226/229 [00:06<00:00, 30.75it/s]

Organizations: 100% 229/229 [00:07<00:00, 32.70it/s]

 Organizations: 457,954 Load Completed


In [24]:
CYPHER_LOCS = """
UNWIND $batch AS r
MERGE (s:Source {name: r.source})
MERGE (l:Location {name: r.loc_name})
  ON CREATE SET l.country_code = r.country, l.lat = r.lat, l.long = r.lng
MERGE (s)-[rel:MENTIONS_LOCATION]->(l)
  ON CREATE SET rel.count = 1
  ON MATCH SET  rel.count = rel.count + 1
"""
 
if not locs_df.empty:
    l_records = locs_df.rename(columns={
        "SourceCommonName":"source", "LocName":"loc_name",
        "CountryCode":"country", "Lat":"lat", "Long":"lng",
    })[["source","loc_name","country","lat","lng"]].fillna("").to_dict("records")
    batch_load(CYPHER_LOCS, l_records, label="Locations")

Locations:   0% 0/578 [00:00<?, ?it/s]

Locations:   0% 1/578 [00:00<00:59,  9.65it/s]

Locations:   1% 4/578 [00:00<00:27, 20.57it/s]

Locations:   1% 7/578 [00:00<00:23, 24.26it/s]

Locations:   2% 10/578 [00:00<00:21, 25.97it/s]

...

Locations:  99% 571/578 [00:21<00:00, 24.86it/s]

Locations:  99% 574/578 [00:22<00:00, 24.80it/s]

Locations: 100% 577/578 [00:22<00:00, 24.50it/s]

Locations: 100% 578/578 [00:22<00:00, 25.97it/s]

 Locations: 1,154,760 Load Completed


### Edges

In [26]:
CYPHER_EVENT_NODES = """
UNWIND $batch AS r
MERGE (:Event {global_event_id: r.eid})
"""
 
CYPHER_MENTIONS = """
UNWIND $batch AS r
MATCH (e:Event {global_event_id: r.eid})
MERGE (s:Source {name: r.source})
MERGE (e)-[rel:MENTIONED_IN]->(s)
  ON CREATE SET rel.confidence = r.conf, rel.tone = r.tone
"""
 
if not mentions.empty:
    m_df = mentions.rename(columns={
        "GlobalEventID":"eid", "MentionSourceName":"source",
        "Confidence":"conf", "MentionDocTone":"tone",
    })[["eid","source","conf","tone"]].fillna("")
 
    eid_records = m_df[["eid"]].drop_duplicates().to_dict("records")
    batch_load(CYPHER_EVENT_NODES, eid_records, label="Event Node")
    batch_load(CYPHER_MENTIONS, m_df.to_dict("records"), label="Mentions")

Event Node:   0% 0/62 [00:00<?, ?it/s]

Event Node:   6% 4/62 [00:00<00:01, 39.52it/s]

Event Node:  24% 15/62 [00:00<00:00, 80.16it/s]

Event Node:  42% 26/62 [00:00<00:00, 90.97it/s]

Event Node:  60% 37/62 [00:00<00:00, 95.14it/s]

...

Mentions:  91% 129/141 [00:03<00:00, 39.53it/s]

Mentions:  95% 134/141 [00:03<00:00, 39.94it/s]

Mentions:  99% 139/141 [00:03<00:00, 40.26it/s]

Mentions: 100% 141/141 [00:03<00:00, 40.09it/s]

 Mentions: 281,703 Load Completed
